In [4]:
import os
import voyageai
from pinecone import Pinecone
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# 1. Setup Clients
vo = voyageai.Client(api_key=os.environ["VOYAGE_API_KEY"])
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
index = pc.Index("japanese-wiki-index")

# 2. Your Test Question (Ask in English!)
query1 =  "What type of vehicle is Hermes from Kino's Journey"
query2 = "Who is the fastest horse in Uma Musume?"
query3 = "Tell me about Rin Tohsaka"
query4 = "Who are the main characters of Kino's Journey?"
query5 = "Who are the main characters of Bunny Drop or Usagi Drop?"
query6 = "Who is the tank in Kino's Journey?"
query = query1

# 3. Embed the query (IMPORTANT: use input_type="query")
query_emb = vo.embed([query], model="voyage-4-lite", input_type="query").embeddings[0]

# 4. Search Pinecone
results = index.query(vector=query_emb, top_k=3, include_metadata=True)

# 5. Show the results
print(f"--- Search Results for: '{query}' ---")
for match in results["matches"]:
    # Here is the 'Title Injection' we talked about!
    print(f"\n[Score: {match.score:.4f}]")
    print(f"SOURCE: {match.metadata['source']}")
    print(f"MEDIA TYPE: {match.metadata['mediatype']}")
    print("-" * 30)
    print(match.metadata['text'][:400] + "...") # Show first 400 chars



# 6. Prepare the Context from Pinecone results
context_list = []
for match in results["matches"]:
    context_list.append(f"Source: {match.metadata['source']}\nContent: {match.metadata['text']}")

context_text = "\n\n---\n\n".join(context_list)

# 7. Create the Prompt
prompt = f"""
Answer in English. You are a helpful assistant. Answer the question based ONLY on the context provided below. If the answer isn't in the context, say you don't know.

Context:
{context_text}

Question: {query}
Answer:"""

# 8. Generate Answer (Example using OpenAI - requires 'openai' library)
response = client.chat.completions.create(
    model="gpt-4o-mini", # Fast and cheap for RAG
    messages=[{"role": "user", "content": prompt}],
    temperature=0
)

print("\n--- FINAL ANSWER ---")
print(response.choices[0].message.content)

--- Search Results for: 'What type of vehicle is Hermes from Kino's Journey' ---

[Score: 0.5623]
SOURCE: C:\wiki_data_json\anime\キノの旅 -the Beautiful World- the Animated Series.json
MEDIA TYPE: anime
------------------------------
声 - 相ヶ瀬龍史 / 斉藤壮馬 / 野田順子（ラジオドラマ）
キノの相棒。
キノの乗物兼話し相手であるモトラド（二輪車）。言葉を話せるが、それ以外は普通の二輪車で自律走行することはできない。少年のような声で喋り、性格は軽妙で人懐っこい。その反面でシリアスな場面でも冗談を言うため、キノから怒られることもある。
機械として科学知識が豊富で計算速度が速く、また視力など人のそれを凌駕した能力を持つ。一方でモトラド故の独特な哲学を持っており、人の死にも動じることがないドライな性格でもある。また、知識が豊富な反面、よく変な間違い方をした慣用句や諺を使い、キノにツッコミを入れられている。もっともキノがツッコミを入れないと不思議がるなど、わざと間違えている。
元は「大人の国」のスクラップで、初代キノに修理された。その後、現在のキノの国外脱出を手伝い、そのまま相棒となった。エルメスという名は、初代キノの昔...

[Score: 0.5623]
SOURCE: C:\wiki_data_json\anime\キノの旅 -the Beautiful World-.json
MEDIA TYPE: anime
------------------------------
声 - 相ヶ瀬龍史 / 斉藤壮馬 / 野田順子（ラジオドラマ）
キノの相棒。
キノの乗物兼話し相手であるモトラド（二輪車）。言葉を話せるが、それ以外は普通の二輪車で自律走行することはできない。少年のような声で喋り、性格は軽妙で人懐っこい。その反面でシリアスな場面でも冗談を言うため、キノから怒られることもある。
機械として科学知識が豊富で計算速度が速く、また視力など人のそれを凌駕した能力を持つ。一方でモトラド故の独特な哲学を持っており、人の死にも動じることがないドライ